# Layer 3: Estimation (class notes)

**What this layer is, in one line:** it combines the Layer 2 map with a few field checks to give each unit one number (the flooded share), a yes/no call, and how sure we are.

These notes explain the ideas. You do not need to know how the code is written.

## 1. The goal

For every unit we want:

1. **theta**: the share of its rice fields that is flooded.
2. **A call**: "above 25 percent" or "below" (the loss line **tau = 0.25**). Every unit always gets a call. There is no "cannot decide".
3. **P_u**: the probability that the call is right about being above the line. A shaky call shows up as a probability near 50 percent, not as a refusal.

## 2. Two imperfect sources of information

| Source | Good | Bad |
|---|---|---|
| **The map** (Layer 2) | covers every field | can be **biased**: consistently too high or too low, especially in a flood it has never seen |
| **Field checks** (an officer visits a spot and sees if it is flooded) | trustworthy | **few**, so noisy, and may be taken from convenient places near roads |

## 3. The main idea: use the map, and let the checks correct it

An analogy. You guess the weight of 100 sacks by eye. Then you weigh 5 properly and learn your eye guesses run 2 kg too high. You subtract 2 kg from all 100 guesses. Five weighings improved all 100 guesses. That beats weighing only 5 sacks and throwing away your guesses.

Same here: the map gives a guess for every field, and the few checks tell us **how wrong the map is on average**. We subtract that error. The reason this works with few checks: the map's *error* varies much less from place to place than flooding itself does, and a steady quantity is easier to estimate from a few samples. This approach is called **prediction-powered inference**.

The map's error for one field is simply: **truth minus map probability**. It is called the **rectifier**. Negative means the map over-reads flooding.

## 4. Grouping fields before correcting

The map's error is not the same everywhere, so fields are grouped by what the map says:

* **Mapped dry**: map probability below 0.15.
* **Uncertain**: 0.15 to 0.6.
* **Mapped flooded**: 0.6 and above.

These cut points were fixed in advance, not tuned. The correction is worked out separately for each group and then combined using how much of the unit each group makes up.

On the training events the map over-reads flooding in every group (average errors of about -0.035, -0.207 and -0.140), mostly in the uncertain group.

## 5. Not many checks? Borrow from neighbours

With 0 or 1 checks in a unit, its own correction is unreliable. So we **shrink** it toward a shared correction, like a student with one test mark being judged partly by the class average. The more checks a unit has, the more we trust its own number and the less we lean on the shared one. The setting that controls how strongly we lean on the shared number was chosen using training events only.

## 6. Fair weighting of checks

If officers check mostly near roads, those checks are not a fair sample. We know each check's chance of being picked (its **inclusion probability**, see Layer 1) and weight it by one divided by that chance. This undoes the unfairness. We tested it with a deliberately road-biased "route" style of checking: an unweighted version was biased by -0.030, a weighted version by -0.011, and the exact "Horvitz-Thompson" form by about +0.002 (very close to zero).

## 7. Neighbouring fields are alike: clustering

Fields in the same small area flood together and are mis-mapped together. So four checks in one spot are worth less than four checks in four different places. If we ignore this we claim more certainty than we have, and our 90 percent ranges end up covering the truth less than 90 percent of the time. We measure how alike neighbours are (the **intra-cluster correlation**, about 0.33 here), which makes four neighbouring checks worth about two independent ones, and compute uncertainty accordingly.

## 8. How sure are we? P_u, and farmer reports

**P_u** comes from the estimate and its uncertainty: the further the estimate is from the 25 percent line, relative to its uncertainty, the closer P_u is to 0 or 1.

**Farmer reports** are never mixed into theta. Farmers who report are mostly the ones who were flooded, so reports lean toward flooding. Instead we compute a **flip value**: how biased the reports would have to be before trusting them changes the call. A large flip value means the call is safe. A small one means it depends on trusting the reports. The farmer reports in our runs are simulated.

## 9. How we tested it, and what happened

We repeated the "draw some random checks, compute estimates" experiment many times by computer on events not used for fitting, comparing against two baselines: **the map alone** and **the checks alone**. The field checks are simulated from the hand labels. No officer was involved.

**Test events** (34 units), share of units given the wrong call:

| Checks per unit | Map alone | Checks alone | **Ours** |
|---|---|---|---|
| 0 | 0.147 | not possible | 0.206 |
| 16 | 0.147 | 0.131 | 0.098 |
| 32 | 0.147 | 0.098 | **0.068** |

* With **32 checks per unit** ours is clearly best: 6.8 percent wrong against 14.7 (map) and 9.8 (checks).
* With **zero checks ours is worse than the raw map** (20.6 against 14.7 percent). The correction learned on training events does not carry over to new test events. We say so openly.
* Our probabilities are honest enough (calibration error 0.041 at 16 checks), but our 90 percent ranges only cover the truth about 81 percent of the time at 16 checks. They are a bit too narrow on new events.
* Test is harder than validation. The gap is real and we report it.

## 10. The honest history

Our **first version** shrank every unit toward the training correction and treated that correction as certain. On its first test run it failed its own coverage target: ranges covered only **74 percent** at 16 checks. Reason: in India the map under-reads flooding, while on the training events it over-reads, so correcting toward the training average pulled estimates the wrong way.

The **second version** (the one we use) allows for how much the correction can differ between flood events, and learns part of it from the checks in hand. **It was written after we saw the first test result, so its test numbers are not a clean, untouched estimate.** The first version's result is kept on file and we always say this.

## 11. Key numbers

| Thing | Value |
|---|---|
| Loss line tau | **0.25** |
| Groups by map probability | below 0.15, 0.15 to 0.6, 0.6 and above |
| Units | 125 train, 27 validation, 34 test (made from 121 chip maps) |
| Neighbour similarity | about 0.33, so 4 nearby checks are worth about 2 |
| Test, 32 checks, wrong calls | ours **0.068**, map **0.147**, checks **0.098** |
| Test, 0 checks | ours **0.206**, map **0.147** (ours worse) |
| Test, 16 checks | ranges cover truth **0.81** of the time; calibration error **0.041** |
| First version | covered only **0.74** at 16 checks |
| Road-biased checks, bias | unweighted **-0.030**, weighted **-0.011**, exact form **+0.002** |

## 12. Glossary

* **Bias**: error that points the same way every time.
* **Rectifier**: truth minus map probability; how wrong the map is.
* **Shrinkage**: leaning on a shared value when a unit has little data.
* **Inclusion probability**: known chance that a place was picked for checking.
* **Cluster**: neighbouring fields that behave alike.
* **Coverage**: how often a stated range (say 90 percent) really contains the truth.
* **P_u**: probability that the unit is above the loss line.
* **Flip value**: how biased farmer reports must be to change a call.
* **Prediction-powered inference**: use a prediction everywhere, and use a few real measurements to correct its error.

## 13. Review questions

1. **What does Layer 3 do?** Gives each unit a flooded share, a call, and a probability, by correcting the map with a few field checks.
2. **Why not just use the map?** It can be biased in a new flood.
3. **Why not just use the checks?** Too few and too noisy.
4. **What is the rectifier?** The map's error: truth minus map probability.
5. **Why does a few checks go a long way?** The map's error is steadier than flooding itself.
6. **Why weight checks?** Checks near roads are not a fair sample; weighting by 1 over inclusion probability fixes that.
7. **Why does clustering matter?** Neighbours are alike, so nearby checks are worth less, and ignoring it overstates our certainty.
8. **Are farmer reports used?** Not in the estimate. They only feed a cross-check (flip value).
9. **What is a weakness?** With zero checks it is worse than the raw map, and its ranges are a little narrow on new events.
10. **What happened with the first version?** It missed its coverage target on test (0.74). The second version fixed it but was written after seeing the test result, so its test numbers are not clean.
11. **Are the field checks real?** No, they are simulated from the hand labels.